In [9]:
from google.colab import drive
drive.mount('/content/drive')

import os
os.environ['QT_QPA_PLATFORM'] = 'offscreen'

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [4]:
!apt-get update -q
!apt-get install colmap -y -q

Get:1 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease [3,632 B]
Get:2 https://cli.github.com/packages stable InRelease [3,917 B]
Hit:3 http://archive.ubuntu.com/ubuntu jammy InRelease
Get:4 https://r2u.stat.illinois.edu/ubuntu jammy InRelease [6,555 B]
Get:5 http://security.ubuntu.com/ubuntu jammy-security InRelease [129 kB]
Get:6 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ Packages [102 kB]
Get:7 http://archive.ubuntu.com/ubuntu jammy-updates InRelease [128 kB]
Get:8 https://cli.github.com/packages stable/main amd64 Packages [356 B]
Get:9 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease [18.1 kB]
Get:10 https://r2u.stat.illinois.edu/ubuntu jammy/main all Packages [10.6 MB]
Hit:11 https://ppa.launchpadcontent.net/ubuntugis/ppa/ubuntu jammy InRelease
Get:12 http://archive.ubuntu.com/ubuntu jammy-backports InRelease [127 kB]
Get:13 http://security.ubuntu.com/ubuntu jammy-security/restricted amd64 Packages [7,479 kB]
Get:14 https:/

In [5]:
!colmap -h

COLMAP 3.7 -- Structure-from-Motion and Multi-View Stereo
              (Commit Unknown on Unknown without CUDA)

Usage:
  colmap [command] [options]

Documentation:
  https://colmap.github.io/

Example usage:
  colmap help [ -h, --help ]
  colmap gui
  colmap gui -h [ --help ]
  colmap automatic_reconstructor -h [ --help ]
  colmap automatic_reconstructor --image_path IMAGES --workspace_path WORKSPACE
  colmap feature_extractor --image_path IMAGES --database_path DATABASE
  colmap exhaustive_matcher --database_path DATABASE
  colmap mapper --image_path IMAGES --database_path DATABASE --output_path MODEL
  ...

Available commands:
  help
  gui
  automatic_reconstructor
  bundle_adjuster
  color_extractor
  database_cleaner
  database_creator
  database_merger
  delaunay_mesher
  exhaustive_matcher
  feature_extractor
  feature_importer
  hierarchical_mapper
  image_deleter
  image_filterer
  image_rectifier
  image_registrator
  image_undistorter
  image_undistorter_standalone
  mapper

In [6]:
import os

image_path = '/content/drive/MyDrive/SpatialAI/data/raw_images'
workspace_path = '/content/drive/MyDrive/SpatialAI/outputs/camera_poses'
database_path = os.path.join(workspace_path, 'database.db')

os.makedirs(workspace_path, exist_ok=True)

In [15]:
!colmap feature_extractor \
    --image_path {image_path} \
    --database_path {database_path} \
    --ImageReader.single_camera 1 \
    --SiftExtraction.use_gpu 0


Feature extraction

Processed file [1/36]
  Name:            frame_0000.jpg
  Dimensions:      478 x 850
  Camera:          #1 - SIMPLE_RADIAL
  Focal Length:    1020.00px
  Features:        1332
Processed file [2/36]
  Name:            frame_0001.jpg
  Dimensions:      478 x 850
  Camera:          #1 - SIMPLE_RADIAL
  Focal Length:    1020.00px
  Features:        1701
Processed file [3/36]
  Name:            frame_0002.jpg
  Dimensions:      478 x 850
  Camera:          #1 - SIMPLE_RADIAL
  Focal Length:    1020.00px
  Features:        1672
Processed file [4/36]
  Name:            frame_0003.jpg
  Dimensions:      478 x 850
  Camera:          #1 - SIMPLE_RADIAL
  Focal Length:    1020.00px
  Features:        1666
Processed file [5/36]
  Name:            frame_0004.jpg
  Dimensions:      478 x 850
  Camera:          #1 - SIMPLE_RADIAL
  Focal Length:    1020.00px
  Features:        1839
Processed file [6/36]
  Name:            frame_0005.jpg
  Dimensions:      478 x 850
  Camera:     

In [16]:
!colmap exhaustive_matcher \
    --database_path {database_path} \
    --SiftMatching.use_gpu 0


Exhaustive feature matching

Matching block [1/1, 1/1] in 156.981s
Elapsed time: 2.618 [minutes]


In [17]:
sparse_path = os.path.join(workspace_path, 'sparse')
os.makedirs(sparse_path, exist_ok=True)

!colmap mapper \
    --database_path {database_path} \
    --image_path {image_path} \
    --output_path {sparse_path}


Loading database

Loading cameras... 1 in 0.000s
Loading matches... 481 in 0.008s
Loading images... 36 in 0.009s (connected 36)
Building correspondence graph... in 0.027s (ignored 0)

Elapsed time: 0.001 [minutes]


Finding good initial image pair


Initializing with image pair #16 and #8


Global bundle adjustment

iter      cost      cost_change  |gradient|   |step|    tr_ratio  tr_radius  ls_iter  iter_time  total_time
   0  5.632390e+01    0.00e+00    1.68e+03   0.00e+00   0.00e+00  1.00e+04        0    6.56e-04    2.04e-03
   1  5.152328e+01    4.80e+00    5.16e+03   1.73e+01   4.44e-01  9.99e+03        1    1.06e-03    3.15e-03
   2  4.364168e+01    7.88e+00    9.64e+02   1.49e+01   9.72e-01  3.00e+04        1    8.10e-04    3.97e-03
   3  4.239963e+01    1.24e+00    2.87e+03   2.45e+01   5.80e-01  3.01e+04        1    8.05e-04    4.79e-03
   4  4.113126e+01    1.27e+00    1.44e+03   1.49e+01   8.96e-01  5.99e+04        1    8.29e-04    5.63e-03
   5  4.118562e+01   -5.44e-02   

In [18]:
sparse_model_path = os.path.join(sparse_path, '0')  # COLMAP saves the result in a subfolder named '0'
text_output_path = os.path.join(workspace_path, 'sparse_text')
os.makedirs(text_output_path, exist_ok=True)

!colmap model_converter \
    --input_path {sparse_model_path} \
    --output_path {text_output_path} \
    --output_type TXT

In [19]:
import numpy as np

def read_images_txt(path):
    cameras = []
    with open(path, 'r') as f:
        lines = f.readlines()

    for line in lines:
        if line.startswith('#') or line.strip() == '':
            continue
        parts = line.split()
        if len(parts) >= 10 and not parts[0].replace('.', '').isdigit() is False:
            try:
                image_id = int(parts[0])
                qw, qx, qy, qz = map(float, parts[1:5])
                tx, ty, tz = map(float, parts[5:8])
                name = parts[9]
                cameras.append({'name': name, 'quat': [qw, qx, qy, qz], 'trans': [tx, ty, tz]})
            except ValueError:
                continue  # this line was the 2D points list, not a pose line, skip it
    return cameras

images_txt_path = os.path.join(text_output_path, 'images.txt')
camera_poses = read_images_txt(images_txt_path)
print(f"Parsed {len(camera_poses)} camera poses.")

Parsed 36 camera poses.


In [20]:
def read_points3D_txt(path):
    points = []
    colors = []
    with open(path, 'r') as f:
        for line in f:
            if line.startswith('#') or line.strip() == '':
                continue
            parts = line.split()
            x, y, z = map(float, parts[1:4])
            r, g, b = map(int, parts[4:7])
            points.append([x, y, z])
            colors.append([r, g, b])
    return np.array(points), np.array(colors)

points3d_path = os.path.join(text_output_path, 'points3D.txt')
points, colors = read_points3D_txt(points3d_path)
print(f"Loaded {len(points)} 3D points.")

Loaded 5234 3D points.


In [21]:
def quaternion_to_camera_center(qw, qx, qy, qz, tx, ty, tz):
    # Build rotation matrix from quaternion
    R = np.array([
        [1 - 2*qy**2 - 2*qz**2, 2*qx*qy - 2*qz*qw, 2*qx*qz + 2*qy*qw],
        [2*qx*qy + 2*qz*qw, 1 - 2*qx**2 - 2*qz**2, 2*qy*qz - 2*qx*qw],
        [2*qx*qz - 2*qy*qw, 2*qy*qz + 2*qx*qw, 1 - 2*qx**2 - 2*qy**2]
    ])
    t = np.array([tx, ty, tz])
    camera_center = -R.T @ t  # this converts COLMAP's stored pose into actual camera position in 3D space
    return camera_center

camera_positions = []
for cam in camera_poses:
    qw, qx, qy, qz = cam['quat']
    tx, ty, tz = cam['trans']
    center = quaternion_to_camera_center(qw, qx, qy, qz, tx, ty, tz)
    camera_positions.append(center)

camera_positions = np.array(camera_positions)
print(f"Computed {len(camera_positions)} camera positions.")

Computed 36 camera positions.


In [22]:
import plotly.graph_objects as go

fig = go.Figure()

# Plot the sparse point cloud, colored using actual image colors
fig.add_trace(go.Scatter3d(
    x=points[:, 0], y=points[:, 1], z=points[:, 2],
    mode='markers',
    marker=dict(size=1.5, color=[f'rgb({r},{g},{b})' for r, g, b in colors]),
    name='Point Cloud'
))

# Plot camera positions
fig.add_trace(go.Scatter3d(
    x=camera_positions[:, 0], y=camera_positions[:, 1], z=camera_positions[:, 2],
    mode='markers',
    marker=dict(size=4, color='red', symbol='diamond'),
    name='Camera Positions'
))

fig.update_layout(
    scene=dict(aspectmode='data'),
    title='Sparse Reconstruction: Point Cloud + Camera Poses',
    width=900, height=700
)
fig.show()

In [23]:
# Distance from each camera to the "center" of the point cloud
scene_center = points.mean(axis=0)
distances = np.linalg.norm(camera_positions - scene_center, axis=1)

print(f"Scene center: {scene_center}")
print(f"Camera distances from center - min: {distances.min():.2f}, max: {distances.max():.2f}, mean: {distances.mean():.2f}")
print(f"Distance variation (std dev): {distances.std():.2f}")

Scene center: [-0.61137451  0.57400373  7.32033675]
Camera distances from center - min: 7.65, max: 8.35, mean: 8.07
Distance variation (std dev): 0.20


In [24]:
def quaternion_to_viewing_direction(qw, qx, qy, qz):
    R = np.array([
        [1 - 2*qy**2 - 2*qz**2, 2*qx*qy - 2*qz*qw, 2*qx*qz + 2*qy*qw],
        [2*qx*qy + 2*qz*qw, 1 - 2*qx**2 - 2*qz**2, 2*qy*qz - 2*qx*qw],
        [2*qx*qz - 2*qy*qw, 2*qy*qz + 2*qx*qw, 1 - 2*qx**2 - 2*qy**2]
    ])
    # Camera looks down its own -Z axis by convention; rotate that into world space
    viewing_dir = R.T @ np.array([0, 0, 1])
    return viewing_dir

fig = go.Figure()

fig.add_trace(go.Scatter3d(
    x=points[:, 0], y=points[:, 1], z=points[:, 2],
    mode='markers',
    marker=dict(size=1.5, color=[f'rgb({r},{g},{b})' for r, g, b in colors]),
    name='Point Cloud'
))

fig.add_trace(go.Scatter3d(
    x=camera_positions[:, 0], y=camera_positions[:, 1], z=camera_positions[:, 2],
    mode='markers',
    marker=dict(size=4, color='red', symbol='diamond'),
    name='Camera Positions'
))

# Draw a short line from each camera showing which way it's pointing
for cam, pos in zip(camera_poses, camera_positions):
    qw, qx, qy, qz = cam['quat']
    direction = quaternion_to_viewing_direction(qw, qx, qy, qz)
    end_point = pos + direction * 1.5  # scale the line for visibility
    fig.add_trace(go.Scatter3d(
        x=[pos[0], end_point[0]], y=[pos[1], end_point[1]], z=[pos[2], end_point[2]],
        mode='lines',
        line=dict(color='orange', width=3),
        showlegend=False
    ))

fig.update_layout(scene=dict(aspectmode='data'), title='Cameras with Viewing Direction', width=900, height=700)
fig.show()